## 1st demo


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

# Device configuration (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
initial_alpha = 0.1
max_alpha = 0.9
initial_beta = 0.05
max_beta = 0.3
learning_rate = 0.001
num_epochs = 10
batch_size = 64

# CIFAR-100 Data Transforms
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Loading CIFAR-100 Dataset
train_dataset = datasets.CIFAR100(root='./data', train=True, transform=transform, download=True)
val_dataset = datasets.CIFAR100(root='./data', train=False, transform=transform, download=True)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

# Defining teacher-student model pairs
model_pairs = {
    "VGG13-MobileNetV2": (models.vgg13_bn(pretrained=True), models.mobilenet_v2(pretrained=False)),
    "ResNet50-MobileNetV2": (models.resnet50(pretrained=True), models.mobilenet_v2(pretrained=False)),
    "ResNet50-ShuffleNetV1": (models.resnet50(pretrained=True), models.shufflenet_v2_x1_0(pretrained=False))
}

# Knowledge Distillation Loss with Entropy Regularization
def custom_knowledge_distillation_loss(student_logits, teacher_logits, true_labels, alpha, beta):
    cross_entropy_loss = F.cross_entropy(student_logits, true_labels)
    teacher_probs = F.softmax(teacher_logits, dim=1)
    student_log_probs = F.log_softmax(student_logits, dim=1)
    distillation_loss = F.kl_div(student_log_probs, teacher_probs, reduction="batchmean")
    student_probs = F.softmax(student_logits, dim=1)
    entropy_regularization = -torch.sum(student_probs * torch.log(student_probs + 1e-8)) / student_probs.size(0)
    total_loss = (1 - alpha) * cross_entropy_loss + alpha * distillation_loss + beta * entropy_regularization
    return total_loss

# Dynamic adjustment for alpha and beta
def update_hyperparameters(epoch, num_epochs):
    alpha = initial_alpha + (max_alpha - initial_alpha) * (epoch / num_epochs)
    beta = initial_beta + (max_beta - initial_beta) * (epoch / num_epochs)
    return alpha, beta

results = []

# Loop over each model pair
for model_name, (teacher_model, student_model) in model_pairs.items():
    # Prepare teacher and student models for CIFAR-100
    if hasattr(teacher_model, 'fc'):  # For models like ResNet
        teacher_model.fc = nn.Linear(teacher_model.fc.in_features, 100)
    elif hasattr(teacher_model, 'classifier'):  # For models like VGG
        teacher_model.classifier[-1] = nn.Linear(teacher_model.classifier[-1].in_features, 100)

    if hasattr(student_model, 'fc'):  # For models like ResNet or MobileNetV2
        student_model.fc = nn.Linear(student_model.fc.in_features, 100)
    elif hasattr(student_model, 'classifier'):  # For models like VGG
        student_model.classifier[-1] = nn.Linear(student_model.classifier[-1].in_features, 100)

    teacher_model = teacher_model.to(device).eval()  # Freeze teacher
    student_model = student_model.to(device)

    optimizer = optim.Adam(student_model.parameters(), lr=learning_rate)

    best_accuracy = 0
    best_alpha = 0
    best_beta = 0
    best_confusion_matrix = None

    print(f"Training with {model_name} pair:")

    # Training Loop
    for epoch in range(num_epochs):
        alpha, beta = update_hyperparameters(epoch, num_epochs)
        student_model.train()
        running_loss = 0.0
        all_train_labels = []
        all_train_preds = []

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            # teacher and student outputs
            with torch.no_grad():
                teacher_logits = teacher_model(inputs)
            student_logits = student_model(inputs)

            # loss and backpropagate
            loss = custom_knowledge_distillation_loss(student_logits, teacher_logits, labels, alpha, beta)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

            # predictions and labels for training accuracy calculation storing
            _, train_preds = torch.max(student_logits, 1)
            all_train_labels.extend(labels.cpu().numpy())
            all_train_preds.extend(train_preds.cpu().numpy())

        # for training accuracy
        train_accuracy = accuracy_score(all_train_labels, all_train_preds)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss / len(train_loader):.4f}, "
              f"Alpha: {alpha:.4f}, Beta: {beta:.4f}, Training Accuracy: {train_accuracy:.4f}")

        # Validation Phase
        student_model.eval()
        all_labels = []
        all_preds = []

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                student_logits = student_model(inputs)
                _, preds = torch.max(student_logits, 1)

                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(preds.cpu().numpy())

        # Calculating Metrics
        accuracy = accuracy_score(all_labels, all_preds)
        precision = precision_score(all_labels, all_preds, average="macro")
        recall = recall_score(all_labels, all_preds, average="macro")

        print(f"Validation - Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}")

        # Update for best results
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_alpha = alpha
            best_beta = beta
            best_confusion_matrix = confusion_matrix(all_labels, all_preds)

    # Saving best model and metrics for this pair
    results.append({
        "model_pair": model_name,
        "best_accuracy": best_accuracy,
        "best_alpha": best_alpha,
        "best_beta": best_beta,
        "confusion_matrix": best_confusion_matrix
    })
    torch.save(student_model.state_dict(), f'best_student_model_{model_name}_CIFAR100.pth')
    print(f"Best Accuracy for {model_name}: {best_accuracy:.4f} (Alpha: {best_alpha}, Beta: {best_beta})")
    print(f"Confusion Matrix for {model_name}:\n{best_confusion_matrix}\n")

Files already downloaded and verified
Files already downloaded and verified


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG13_BN_Weights.IMAGENET1K_V1`. You can also use `weights=VGG13_BN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: A

Training with VGG13-MobileNetV2 pair:
Epoch [1/10], Loss: 3.6034, Alpha: 0.1000, Beta: 0.0500, Training Accuracy: 0.1275
Validation - Accuracy: 0.1949, Precision: 0.2285, Recall: 0.1949


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch [2/10], Loss: 2.7998, Alpha: 0.1800, Beta: 0.0750, Training Accuracy: 0.2901
Validation - Accuracy: 0.3542, Precision: 0.3967, Recall: 0.3542
Epoch [3/10], Loss: 2.3984, Alpha: 0.2600, Beta: 0.1000, Training Accuracy: 0.4125
Validation - Accuracy: 0.4567, Precision: 0.4915, Recall: 0.4567
Epoch [4/10], Loss: 2.1794, Alpha: 0.3400, Beta: 0.1250, Training Accuracy: 0.4931
Validation - Accuracy: 0.5083, Precision: 0.5367, Recall: 0.5083
Epoch [5/10], Loss: 2.0386, Alpha: 0.4200, Beta: 0.1500, Training Accuracy: 0.5556
Validation - Accuracy: 0.5248, Precision: 0.5635, Recall: 0.5248
Epoch [6/10], Loss: 1.9379, Alpha: 0.5000, Beta: 0.1750, Training Accuracy: 0.6069
Validation - Accuracy: 0.5462, Precision: 0.5765, Recall: 0.5462
Epoch [7/10], Loss: 1.8656, Alpha: 0.5800, Beta: 0.2000, Training Accuracy: 0.6561
Validation - Accuracy: 0.5558, Precision: 0.5925, Recall: 0.5558
Epoch [8/10], Loss: 1.8096, Alpha: 0.6600, Beta: 0.2250, Training Accuracy: 0.6933
Validation - Accuracy: 0.5814

...

Epoch [3/10], Loss: 2.4466, Alpha: 0.2600, Beta: 0.1000, Training Accuracy: 0.3990
Validation - Accuracy: 0.4331, Precision: 0.4613, Recall: 0.4331
Epoch [4/10], Loss: 2.2154, Alpha: 0.3400, Beta: 0.1250, Training Accuracy: 0.4805
Validation - Accuracy: 0.4805, Precision: 0.5252, Recall: 0.4805
Epoch [5/10], Loss: 2.0622, Alpha: 0.4200, Beta: 0.1500, Training Accuracy: 0.5453
Validation - Accuracy: 0.5192, Precision: 0.5478, Recall: 0.5192
Epoch [6/10], Loss: 1.9615, Alpha: 0.5000, Beta: 0.1750, Training Accuracy: 0.5950
Validation - Accuracy: 0.5529, Precision: 0.5848, Recall: 0.5529
Epoch [7/10], Loss: 1.8827, Alpha: 0.5800, Beta: 0.2000, Training Accuracy: 0.6422
Validation - Accuracy: 0.5712, Precision: 0.5955, Recall: 0.5712
Epoch [8/10], Loss: 1.8217, Alpha: 0.6600, Beta: 0.2250, Training Accuracy: 0.6846
Validation - Accuracy: 0.5840, Precision: 0.6053, Recall: 0.5840
Epoch [9/10], Loss: 1.7690, Alpha: 0.7400, Beta: 0.2500, Training Accuracy: 0.7215
Validation - Accuracy: 0.5926

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch [2/10], Loss: 2.8418, Alpha: 0.1800, Beta: 0.0750, Training Accuracy: 0.2884
Validation - Accuracy: 0.3482, Precision: 0.3777, Recall: 0.3482
Epoch [3/10], Loss: 2.4348, Alpha: 0.2600, Beta: 0.1000, Training Accuracy: 0.4092
Validation - Accuracy: 0.4228, Precision: 0.4765, Recall: 0.4228


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch [4/10], Loss: 2.2067, Alpha: 0.3400, Beta: 0.1250, Training Accuracy: 0.4908
Validation - Accuracy: 0.4727, Precision: 0.5075, Recall: 0.4727
Epoch [5/10], Loss: 2.0611, Alpha: 0.4200, Beta: 0.1500, Training Accuracy: 0.5538
Validation - Accuracy: 0.5099, Precision: 0.5486, Recall: 0.5099
Epoch [6/10], Loss: 1.9583, Alpha: 0.5000, Beta: 0.1750, Training Accuracy: 0.6085
Validation - Accuracy: 0.5346, Precision: 0.5578, Recall: 0.5346
Epoch [7/10], Loss: 1.8782, Alpha: 0.5800, Beta: 0.2000, Training Accuracy: 0.6567
Validation - Accuracy: 0.5421, Precision: 0.5582, Recall: 0.5421
Epoch [8/10], Loss: 1.8186, Alpha: 0.6600, Beta: 0.2250, Training Accuracy: 0.6995
Validation - Accuracy: 0.5530, Precision: 0.5707, Recall: 0.5530
Epoch [9/10], Loss: 1.7654, Alpha: 0.7400, Beta: 0.2500, Training Accuracy: 0.7391
Validation - Accuracy: 0.5524, Precision: 0.5774, Recall: 0.5524
Epoch [10/10], Loss: 1.7138, Alpha: 0.8200, Beta: 0.2750, Training Accuracy: 0.7804
Validation - Accuracy: 0.556

Improvment accuracy


## 2nd Demo

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
import itertools

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
initial_alpha = 0.1
max_alpha = 0.9
initial_beta = 0.05
max_beta = 0.3
initial_temperature = 1
max_temperature = 5
learning_rate = 0.001
num_epochs = 50
batch_size = 64

# CIFAR-100 Data Transforms
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

transform_val = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load CIFAR-100 Dataset
dataset = datasets.CIFAR100(root='./data', train=True, transform=transform_train, download=True)

# Split Dataset (90% training, 10% validation)
train_size = 45000
val_size = 5000
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)


train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

# Knowledge Distillation Loss with Temperature Scaling
def custom_knowledge_distillation_loss(student_logits, teacher_logits, true_labels, alpha, beta, temperature):
    cross_entropy_loss = F.cross_entropy(student_logits, true_labels)
    teacher_probs = F.softmax(teacher_logits / temperature, dim=1)
    student_log_probs = F.log_softmax(student_logits / temperature, dim=1)
    distillation_loss = F.kl_div(student_log_probs, teacher_probs, reduction="batchmean") * (temperature ** 2)
    student_probs = F.softmax(student_logits, dim=1)
    entropy_regularization = -torch.sum(student_probs * torch.log(student_probs + 1e-8)) / student_probs.size(0)
    total_loss = (1 - alpha) * cross_entropy_loss + alpha * distillation_loss + beta * entropy_regularization
    return total_loss

# Dynamic adjustment for alpha, beta, and temperature
def update_hyperparameters(epoch, num_epochs):
    alpha = initial_alpha + (max_alpha - initial_alpha) * (epoch / num_epochs)
    beta = initial_beta + (max_beta - initial_beta) * (epoch / num_epochs)
    temperature = initial_temperature + (max_temperature - initial_temperature) * (epoch / num_epochs)
    return alpha, beta, temperature

# Grid Search for Hyperparameter Tuning
alpha_values = [0.1, 0.3, 0.5]
beta_values = [0.05, 0.1, 0.2]
temperature_values = [1, 2, 3]
grid_search_params = list(itertools.product(alpha_values, beta_values, temperature_values))

# Define teacher-student model pairs
model_pairs = {
    #"VGG13-MobileNetV2": (models.vgg13_bn(pretrained=True), models.mobilenet_v2(pretrained=False)),
    #"ResNet50-MobileNetV2": (models.resnet50(pretrained=True), models.mobilenet_v2(pretrained=False)),
    "ResNet50-ShuffleNetV1": (models.resnet50(pretrained=True), models.shufflenet_v2_x1_0(pretrained=False))
}

# Train and Evaluate Models
for model_name, (teacher_model, student_model) in model_pairs.items():
    # Update the classifier layers
    teacher_model.fc = nn.Linear(teacher_model.fc.in_features, 100)
    student_model.fc = nn.Linear(student_model.fc.in_features, 100)

    teacher_model = teacher_model.to(device).eval()
    student_model = student_model.to(device)

    for alpha, beta, temperature in grid_search_params:
        optimizer = optim.AdamW(student_model.parameters(), lr=learning_rate)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

        print(f"Training with {model_name} | Alpha: {alpha}, Beta: {beta}, Temp: {temperature}")
        for epoch in range(num_epochs):
            student_model.train()
            running_loss = 0.0

            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.no_grad():
                    teacher_logits = teacher_model(inputs)
                student_logits = student_model(inputs)

                loss = custom_knowledge_distillation_loss(student_logits, teacher_logits, labels, alpha, beta, temperature)
                loss.backward()
                optimizer.step()
                running_loss += loss.item()

            print(f"Epoch [{epoch+1}/{num_epochs}] | Loss: {running_loss / len(train_loader):.4f}")

            scheduler.step()


        # Validation Metrics
        student_model.eval()
        all_labels = []
        all_preds = []

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                student_logits = student_model(inputs)
                _, preds = torch.max(student_logits, 1)
                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(preds.cpu().numpy())

        accuracy = accuracy_score(all_labels, all_preds)
        print(f"Validation Accuracy: {accuracy:.4f}")



Files already downloaded and verified
Training with ResNet50-ShuffleNetV1 | Alpha: 0.1, Beta: 0.05, Temp: 1
Epoch [1/50] | Loss: 3.8569
Epoch [2/50] | Loss: 3.2899
Epoch [3/50] | Loss: 2.8763
Epoch [4/50] | Loss: 2.5661
Epoch [5/50] | Loss: 2.3366
Epoch [6/50] | Loss: 2.1658
Epoch [7/50] | Loss: 2.0254
Epoch [8/50] | Loss: 1.9129
Epoch [9/50] | Loss: 1.8235
Epoch [10/50] | Loss: 1.7413
Epoch [11/50] | Loss: 1.6705
Epoch [12/50] | Loss: 1.6016
Epoch [13/50] | Loss: 1.5325
Epoch [14/50] | Loss: 1.4787
Epoch [15/50] | Loss: 1.4267
Epoch [16/50] | Loss: 1.3731
Epoch [17/50] | Loss: 1.3318
Epoch [18/50] | Loss: 1.2856
Epoch [19/50] | Loss: 1.2442
Epoch [20/50] | Loss: 1.2027
Epoch [21/50] | Loss: 1.1620
Epoch [22/50] | Loss: 1.1282
Epoch [23/50] | Loss: 1.0846
Epoch [24/50] | Loss: 1.0545
Epoch [25/50] | Loss: 1.0187
Epoch [26/50] | Loss: 0.9923
Epoch [27/50] | Loss: 0.9587
Epoch [28/50] | Loss: 0.9304
Epoch [29/50] | Loss: 0.9089
Epoch [30/50] | Loss: 0.8783
Epoch [31/50] | Loss: 0.8525
Ep

# *Teacher* Model ResNet50


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
import time
import copy

# ==============================================
# Device Configuration
# ==============================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==============================================
# Hyperparameters
# ==============================================
learning_rate = 0.001
num_epochs = 50  # Adjust based on your computational resources
batch_size = 64

# ==============================================
# Data Preparation
# ==============================================
# CIFAR-100 Data Transforms
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.Resize(224),  # Resize to match pre-trained model input size
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Normalize to [-1, 1]
])

transform_val = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load CIFAR-100 Dataset
dataset = datasets.CIFAR100(root='./data', train=True, transform=transform_train, download=True)

# Split Dataset (90% training, 10% validation)
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Apply validation transforms to validation dataset
val_dataset.dataset.transform = transform_val

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

# ==============================================
# Model Definition
# ==============================================

def modify_final_layer(model, num_classes=100):
    """
    Modify the final layer of a pre-trained model to match the number of classes.
    Supports ResNet, ShuffleNet, MobileNet, and VGG architectures.
    """
    if isinstance(model, models.ResNet):
        num_features = model.fc.in_features
        model.fc = nn.Linear(num_features, num_classes)
    elif isinstance(model, models.ShufflenetV2):
        num_features = model.fc.in_features
        model.fc = nn.Linear(num_features, num_classes)
    elif isinstance(model, models.MobileNetV2):
        num_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(num_features, num_classes)
    elif isinstance(model, models.VGG):
        num_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(num_features, num_classes)
    else:
        raise NotImplementedError("Model architecture not supported for modification.")
    return model

# Initialize the teacher model (e.g., ResNet50)
teacher_model_name = "ResNet50"
teacher_model = models.resnet50(pretrained=True)
teacher_model = modify_final_layer(teacher_model, num_classes=100)
teacher_model = teacher_model.to(device)

# ==============================================
# Training Function
# ==============================================

def train_model(model, dataloaders, criterion, optimizer, scheduler, num_epochs=25):
    """
    Train the model and return the best model based on validation accuracy.
    """
    since = time.time()

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        print("-" * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
                dataloader = dataloaders['train']
            else:
                model.eval()   # Set model to evaluate mode
                dataloader = dataloaders['val']

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data
            for inputs, labels in dataloader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Zero the parameter gradients
                optimizer.zero_grad()

                # Forward pass
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # Backward pass and optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # Statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_acc = running_corrects.double() / len(dataloader.dataset)

            print(f"{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

            # Deep copy the model if it has better accuracy
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print(f"Training complete in {int(time_elapsed // 60)}m {int(time_elapsed % 60)}s")
    print(f"Best Validation Acc: {best_acc:.4f}")

    # Load best model weights
    model.load_state_dict(best_model_wts)
    return model

# ==============================================
# Evaluation Function
# ==============================================

def evaluate_model(model, dataloader):
    """
    Evaluate the model on the validation set and print metrics.
    """
    model.eval()
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    conf_matrix = confusion_matrix(all_labels, all_preds)

    print(f"Validation Accuracy : {accuracy:.4f}")
    print(f"Validation Precision: {precision:.4f}")
    print(f"Validation Recall   : {recall:.4f}")
    print("Confusion Matrix:")
    print(conf_matrix)

    return accuracy, precision, recall, conf_matrix

# ==============================================
# Training the Teacher Model
# ==============================================

print("=== Training the Teacher Model ===\n")

# Define loss function, optimizer, and learning rate scheduler
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(teacher_model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

# Define dataloaders
dataloaders = {
    'train': train_loader,
    'val': val_loader
}

# Train the teacher model
teacher_model = train_model(teacher_model, dataloaders, criterion, optimizer, scheduler, num_epochs=num_epochs)

# ==============================================
# Evaluating the Teacher Model
# ==============================================

print("\n=== Evaluating the Teacher Model ===\n")
accuracy, precision, recall, conf_matrix = evaluate_model(teacher_model, val_loader)

print("\n=== Training and Evaluation Completed ===")


Using device: cuda


100%|██████████| 169M/169M [00:03<00:00, 42.4MB/s]


Extracting ./data/cifar-100-python.tar.gz to ./data


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 195MB/s]


=== Training the Teacher Model ===

Epoch 1/50
----------
Train Loss: 2.8872 Acc: 0.2629
Val Loss: 2.4842 Acc: 0.3532

Epoch 2/50
----------
Train Loss: 1.8659 Acc: 0.4751
Val Loss: 1.7774 Acc: 0.4968

Epoch 3/50
----------
Train Loss: 1.4375 Acc: 0.5840
Val Loss: 1.5734 Acc: 0.5518

Epoch 4/50
----------
Train Loss: 1.1525 Acc: 0.6587
Val Loss: 1.3901 Acc: 0.6038

Epoch 5/50
----------
Train Loss: 0.9320 Acc: 0.7142
Val Loss: 1.3737 Acc: 0.6184

Epoch 6/50
----------
Train Loss: 0.7257 Acc: 0.7733
Val Loss: 1.4043 Acc: 0.6240

Epoch 7/50
----------
Train Loss: 0.5509 Acc: 0.8245
Val Loss: 1.4267 Acc: 0.6270

Epoch 8/50
----------
Train Loss: 0.4484 Acc: 0.8578
Val Loss: 1.4160 Acc: 0.6338

Epoch 9/50
----------
Train Loss: 0.3012 Acc: 0.9021
Val Loss: 1.4524 Acc: 0.6418

Epoch 10/50
----------
Train Loss: 0.2355 Acc: 0.9237
Val Loss: 1.5230 Acc: 0.6436

Epoch 11/50
----------
Train Loss: 0.2022 Acc: 0.9333
Val Loss: 1.5633 Acc: 0.6400

Epoch 12/50
----------
Train Loss: 0.1623 Acc: 0.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
import time
import copy

# ==============================================
# Device Configuration
# ==============================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==============================================
# Hyperparameters
# ==============================================
learning_rate = 0.001
num_epochs = 50  # Adjust based on your computational resources
batch_size = 64

# ==============================================
# Data Preparation
# ==============================================
# CIFAR-100 Data Transforms
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.Resize(224),  # Resize to match pre-trained model input size
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Normalize to [-1, 1]
])

transform_val = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load CIFAR-100 Dataset
dataset = datasets.CIFAR100(root='./data', train=True, transform=transform_train, download=True)

# Verify the total number of samples
print(f"Total number of samples in the dataset: {len(dataset)}")

# Split Dataset (e.g., 90% training, 10% validation)
train_size = int(0.9 * len(dataset))  # 45,000
val_size = len(dataset) - train_size  # 5,000
print(f"Training size: {train_size}")
print(f"Validation size: {val_size}")

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Apply validation transforms to validation dataset
val_dataset.dataset.transform = transform_val

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

# ==============================================
# Model Definition
# ==============================================

def modify_final_layer(model, num_classes=100):
    """
    Modify the final layer of a pre-trained VGG model to match the number of classes.
    """
    if isinstance(model, models.VGG):
        num_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(num_features, num_classes)
    else:
        raise NotImplementedError("Only VGG architectures are supported in this script.")
    return model

# Initialize the teacher model (VGG13-BN)
teacher_model_name = "VGG13-BN"
teacher_model = models.vgg13_bn(pretrained=True)
teacher_model = modify_final_layer(teacher_model, num_classes=100)
teacher_model = teacher_model.to(device)

# ==============================================
# Training Function
# ==============================================

def train_model(model, dataloaders, criterion, optimizer, scheduler, num_epochs=50):
    """
    Train the model and return the best model based on validation accuracy.
    """
    since = time.time()

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        print("-" * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
                dataloader = dataloaders['train']
            else:
                model.eval()   # Set model to evaluate mode
                dataloader = dataloaders['val']

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data
            for inputs, labels in dataloader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Zero the parameter gradients
                optimizer.zero_grad()

                # Forward pass
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # Backward pass and optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # Statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_acc = running_corrects.double() / len(dataloader.dataset)

            print(f"{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

            # Deep copy the model if it has better accuracy
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print(f"Training complete in {int(time_elapsed // 60)}m {int(time_elapsed % 60)}s")
    print(f"Best Validation Acc: {best_acc:.4f}")

    # Load best model weights
    model.load_state_dict(best_model_wts)
    return model

# ==============================================
# Evaluation Function
# ==============================================

def evaluate_model(model, dataloader):
    """
    Evaluate the model on the validation set and print metrics.
    """
    model.eval()
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    conf_matrix = confusion_matrix(all_labels, all_preds)

    print(f"Validation Accuracy : {accuracy:.4f}")
    print(f"Validation Precision: {precision:.4f}")
    print(f"Validation Recall   : {recall:.4f}")
    print("Confusion Matrix:")
    print(conf_matrix)

    return accuracy, precision, recall, conf_matrix

# ==============================================
# Training the Teacher Model
# ==============================================

print("=== Training the Teacher Model ===\n")

# Define loss function, optimizer, and learning rate scheduler
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(teacher_model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

# Define dataloaders
dataloaders = {
    'train': train_loader,
    'val': val_loader
}

# Train the teacher model
teacher_model = train_model(teacher_model, dataloaders, criterion, optimizer, scheduler, num_epochs=num_epochs)

# ==============================================
# Evaluating the Teacher Model
# ==============================================

print("\n=== Evaluating the Teacher Model ===\n")
accuracy, precision, recall, conf_matrix = evaluate_model(teacher_model, val_loader)

print("\n=== Training and Evaluation Completed ===")


Using device: cuda


100%|██████████| 169M/169M [00:19<00:00, 8.58MB/s]


Extracting ./data/cifar-100-python.tar.gz to ./data
Total number of samples in the dataset: 50000
Training size: 45000
Validation size: 5000


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG13_BN_Weights.IMAGENET1K_V1`. You can also use `weights=VGG13_BN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg13_bn-abd245e5.pth" to /root/.cache/torch/hub/checkpoints/vgg13_bn-abd245e5.pth
100%|██████████| 508M/508M [00:02<00:00, 221MB/s]


=== Training the Teacher Model ===

Epoch 1/50
----------
Train Loss: 3.1903 Acc: 0.2103
Val Loss: 2.4977 Acc: 0.3472

Epoch 2/50
----------
Train Loss: 2.2706 Acc: 0.3917
Val Loss: 2.1093 Acc: 0.4350

Epoch 3/50
----------
Train Loss: 1.8489 Acc: 0.4884
Val Loss: 1.8731 Acc: 0.4948

Epoch 4/50
----------
Train Loss: 1.5118 Acc: 0.5713
Val Loss: 1.7926 Acc: 0.5146

Epoch 5/50
----------
Train Loss: 1.2210 Acc: 0.6461
Val Loss: 1.7486 Acc: 0.5314

Epoch 6/50
----------
Train Loss: 0.9687 Acc: 0.7125
Val Loss: 1.7994 Acc: 0.5292

Epoch 7/50
----------
Train Loss: 0.7513 Acc: 0.7725
Val Loss: 1.7151 Acc: 0.5620

Epoch 8/50
----------
Train Loss: 0.5942 Acc: 0.8189
Val Loss: 1.8238 Acc: 0.5510

Epoch 9/50
----------
Train Loss: 0.4806 Acc: 0.8517
Val Loss: 1.8743 Acc: 0.5576

Epoch 10/50
----------
Train Loss: 0.3974 Acc: 0.8762
Val Loss: 1.9995 Acc: 0.5586

Epoch 11/50
----------
Train Loss: 0.3456 Acc: 0.8929
Val Loss: 1.9309 Acc: 0.5640

Epoch 12/50
----------
Train Loss: 0.3171 Acc: 0.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
import time
import copy

# ==============================================
# Device Configuration
# ==============================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==============================================
# Hyperparameters
# ==============================================
learning_rate = 0.001
num_epochs = 50  # Adjust based on your computational resources
batch_size = 64

# ==============================================
# Data Preparation
# ==============================================
# CIFAR-100 Data Transforms
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.Resize(224),  # Resize to match pre-trained model input size
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Normalize to [-1, 1]
])

transform_val = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load CIFAR-100 Dataset
dataset = datasets.CIFAR100(root='./data', train=True, transform=transform_train, download=True)

# Verify the total number of samples
print(f"Total number of samples in the dataset: {len(dataset)}")

# Split Dataset (e.g., 90% training, 10% validation)
train_size = int(0.9 * len(dataset))  # 45,000
val_size = len(dataset) - train_size  # 5,000
print(f"Training size: {train_size}")
print(f"Validation size: {val_size}")

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Apply validation transforms to validation dataset
val_dataset.dataset.transform = transform_val

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

# ==============================================
# Model Definition
# ==============================================

def modify_final_layer(model, num_classes=100):
    """
    Modify the final layer of a pre-trained VGG model to match the number of classes.
    """
    if isinstance(model, models.VGG):
        num_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(num_features, num_classes)
    else:
        raise NotImplementedError("Only VGG architectures are supported in this script.")
    return model

# Initialize the teacher model (VGG13)
teacher_model_name = "VGG13"
teacher_model = models.vgg13(pretrained=True)
teacher_model = modify_final_layer(teacher_model, num_classes=100)
teacher_model = teacher_model.to(device)

# ==============================================
# Training Function
# ==============================================

def train_model(model, dataloaders, criterion, optimizer, scheduler, num_epochs=50):
    """
    Train the model and return the best model based on validation accuracy.
    """
    since = time.time()

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        print("-" * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
                dataloader = dataloaders['train']
            else:
                model.eval()   # Set model to evaluate mode
                dataloader = dataloaders['val']

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data
            for inputs, labels in dataloader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Zero the parameter gradients
                optimizer.zero_grad()

                # Forward pass
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # Backward pass and optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # Statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_acc = running_corrects.double() / len(dataloader.dataset)

            print(f"{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

            # Deep copy the model if it has better accuracy
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print(f"Training complete in {int(time_elapsed // 60)}m {int(time_elapsed % 60)}s")
    print(f"Best Validation Acc: {best_acc:.4f}")

    # Load best model weights
    model.load_state_dict(best_model_wts)
    return model

# ==============================================
# Evaluation Function
# ==============================================

def evaluate_model(model, dataloader):
    """
    Evaluate the model on the validation set and print metrics.
    """
    model.eval()
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    conf_matrix = confusion_matrix(all_labels, all_preds)

    print(f"Validation Accuracy : {accuracy:.4f}")
    print(f"Validation Precision: {precision:.4f}")
    print(f"Validation Recall   : {recall:.4f}")
    print("Confusion Matrix:")
    print(conf_matrix)

    return accuracy, precision, recall, conf_matrix

# ==============================================
# Training the Teacher Model
# ==============================================

print("=== Training the Teacher Model ===\n")

# Define loss function, optimizer, and learning rate scheduler
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(teacher_model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

# Define dataloaders
dataloaders = {
    'train': train_loader,
    'val': val_loader
}

# Train the teacher model
teacher_model = train_model(teacher_model, dataloaders, criterion, optimizer, scheduler, num_epochs=num_epochs)

# ==============================================
# Evaluating the Teacher Model
# ==============================================

print("\n=== Evaluating the Teacher Model ===\n")
accuracy, precision, recall, conf_matrix = evaluate_model(teacher_model, val_loader)

print("\n=== Training and Evaluation Completed ===")


Using device: cuda


100%|██████████| 169M/169M [00:13<00:00, 12.2MB/s]


Extracting ./data/cifar-100-python.tar.gz to ./data
Total number of samples in the dataset: 50000
Training size: 45000
Validation size: 5000


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG13_Weights.IMAGENET1K_V1`. You can also use `weights=VGG13_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg13-19584684.pth" to /root/.cache/torch/hub/checkpoints/vgg13-19584684.pth
100%|██████████| 508M/508M [00:02<00:00, 226MB/s]


=== Training the Teacher Model ===

Epoch 1/50
----------
Train Loss: 4.1858 Acc: 0.0522
Val Loss: 3.8269 Acc: 0.1076

Epoch 2/50
----------
Train Loss: 3.6259 Acc: 0.1393
Val Loss: 3.4281 Acc: 0.1766

Epoch 3/50
----------
Train Loss: 3.3364 Acc: 0.1918
Val Loss: 3.2845 Acc: 0.2034

Epoch 4/50
----------
Train Loss: 3.1090 Acc: 0.2354
Val Loss: 3.0193 Acc: 0.2634

Epoch 5/50
----------
Train Loss: 2.8937 Acc: 0.2740
Val Loss: 2.8812 Acc: 0.2830

Epoch 6/50
----------
Train Loss: 2.6740 Acc: 0.3190
Val Loss: 2.7976 Acc: 0.3004

Epoch 7/50
----------
Train Loss: 2.4576 Acc: 0.3619
Val Loss: 2.6939 Acc: 0.3262

Epoch 8/50
----------
Train Loss: 2.2568 Acc: 0.4058
Val Loss: 2.6600 Acc: 0.3300

Epoch 9/50
----------
Train Loss: 2.0202 Acc: 0.4584
Val Loss: 2.6225 Acc: 0.3518

Epoch 10/50
----------
Train Loss: 1.7973 Acc: 0.5112
Val Loss: 2.6740 Acc: 0.3472

Epoch 11/50
----------
Train Loss: 1.5936 Acc: 0.5606
Val Loss: 2.7302 Acc: 0.3478

Epoch 12/50
----------
Train Loss: 1.3725 Acc: 0.

KeyboardInterrupt: 

# VGG

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import accuracy_score

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
learning_rate = 0.001
num_epochs = 50
batch_size = 64

# CIFAR-100 Data Transforms
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

transform_val = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load CIFAR-100 Dataset
train_dataset = datasets.CIFAR100(root='./data', train=True, download=True, transform=transform_train)
val_dataset = datasets.CIFAR100(root='./data', train=False, download=True, transform=transform_val)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

# Define and Modify Teacher Model
teacher_model = models.vgg13_bn(pretrained=True)
teacher_model.classifier[-1] = nn.Linear(teacher_model.classifier[-1].in_features, 100)
teacher_model = teacher_model.to(device)

# Optimizer and Loss Function
optimizer = optim.Adam(teacher_model.parameters(), lr=learning_rate)
loss_function = nn.CrossEntropyLoss()

# Train and Validate Model
best_validation_accuracy = 0

for epoch in range(num_epochs):
    # Training
    teacher_model.train()
    total_train_loss = 0
    correct_train = 0
    total_train = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = teacher_model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    train_accuracy = 100 * correct_train / total_train
    train_loss = total_train_loss / len(train_loader)

    # Validation
    teacher_model.eval()
    total_val_loss = 0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = teacher_model(inputs)
            loss = loss_function(outputs, labels)

            total_val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_accuracy = 100 * correct_val / total_val
    val_loss = total_val_loss / len(val_loader)

    # Check for Best Validation Accuracy
    if val_accuracy > best_validation_accuracy:
        best_validation_accuracy = val_accuracy

    print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%")

print(f"Best Validation Accuracy: {best_validation_accuracy:.2f}%")


Files already downloaded and verified
Files already downloaded and verified
Epoch [1/50], Train Loss: 3.3278, Train Accuracy: 17.97%, Val Loss: 2.5947, Val Accuracy: 32.18%
Epoch [2/50], Train Loss: 2.5515, Train Accuracy: 32.51%, Val Loss: 2.1275, Val Accuracy: 41.70%
Epoch [3/50], Train Loss: 2.2462, Train Accuracy: 39.19%, Val Loss: 1.9327, Val Accuracy: 47.25%
Epoch [4/50], Train Loss: 2.0391, Train Accuracy: 44.15%, Val Loss: 1.7574, Val Accuracy: 50.93%
Epoch [5/50], Train Loss: 1.8816, Train Accuracy: 48.03%, Val Loss: 1.6317, Val Accuracy: 54.28%
Epoch [6/50], Train Loss: 1.7601, Train Accuracy: 50.93%, Val Loss: 1.6011, Val Accuracy: 55.95%
Epoch [7/50], Train Loss: 1.6600, Train Accuracy: 53.28%, Val Loss: 1.5547, Val Accuracy: 56.55%
Epoch [8/50], Train Loss: 1.5822, Train Accuracy: 55.46%, Val Loss: 1.4717, Val Accuracy: 57.98%
Epoch [9/50], Train Loss: 1.4973, Train Accuracy: 57.20%, Val Loss: 1.4034, Val Accuracy: 60.57%
Epoch [10/50], Train Loss: 1.4372, Train Accuracy: 

# 3rd demo(3rd model)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
import itertools
import random

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
initial_alpha = 0.1
max_alpha = 0.9
initial_beta = 0.05
max_beta = 0.3
initial_temperature = 1
max_temperature = 5
learning_rate = 0.001
num_epochs = 50
batch_size = 64

# CIFAR-100 Data Transforms
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

transform_val = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load CIFAR-100 Dataset
dataset = datasets.CIFAR100(root='./data', train=True, transform=transform_train, download=True)

# Split Dataset (90% training, 10% validation)
train_size = 45000
val_size = 5000
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

# Knowledge Distillation Loss with Temperature Scaling
def custom_knowledge_distillation_loss(student_logits, teacher_logits, true_labels, alpha, beta, temperature):
    cross_entropy_loss = F.cross_entropy(student_logits, true_labels)
    teacher_probs = F.softmax(teacher_logits / temperature, dim=1)
    student_log_probs = F.log_softmax(student_logits / temperature, dim=1)
    distillation_loss = F.kl_div(student_log_probs, teacher_probs, reduction="batchmean") * (temperature ** 2)
    student_probs = F.softmax(student_logits, dim=1)
    entropy_regularization = -torch.sum(student_probs * torch.log(student_probs + 1e-8)) / student_probs.size(0)
    total_loss = (1 - alpha) * cross_entropy_loss + alpha * distillation_loss + beta * entropy_regularization
    return total_loss

# Random Search for Temperature                          #here try with grid search
def random_search_temperature(num_samples):
    return [random.uniform(initial_temperature, max_temperature) for _ in range(num_samples)]

# Grid Search for Hyperparameter Tuning
alpha_values = [0.1, 0.3, 0.5]
beta_values = [0.05, 0.1, 0.2]
grid_search_params = list(itertools.product(alpha_values, beta_values))

# Define teacher-student model pairs
model_pairs = {
    "ResNet50-ShuffleNetV1": (models.resnet50(pretrained=True), models.shufflenet_v2_x1_0(pretrained=False))
}

# Train and Evaluate Models
for model_name, (teacher_model, student_model) in model_pairs.items():
    # Update the classifier layers
    teacher_model.fc = nn.Linear(teacher_model.fc.in_features, 100)
    student_model.fc = nn.Linear(student_model.fc.in_features, 100)

    teacher_model = teacher_model.to(device).eval()
    student_model = student_model.to(device)

    # Perform random search for temperature
    num_random_samples = 10
    temperature_candidates = random_search_temperature(num_random_samples)

    for temperature in temperature_candidates:
        print(f"Random Search | Model: {model_name} | Temp: {temperature:.2f}")

        for alpha, beta in grid_search_params:
            optimizer = optim.AdamW(student_model.parameters(), lr=learning_rate)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

            print(f"Grid Search | Alpha: {alpha}, Beta: {beta}, Temp: {temperature:.2f}")
            for epoch in range(num_epochs):
                student_model.train()
                running_loss = 0.0

                for inputs, labels in train_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    optimizer.zero_grad()

                    with torch.no_grad():
                        teacher_logits = teacher_model(inputs)
                    student_logits = student_model(inputs)

                    loss = custom_knowledge_distillation_loss(student_logits, teacher_logits, labels, alpha, beta, temperature)
                    loss.backward()
                    optimizer.step()
                    running_loss += loss.item()

                print(f"Epoch [{epoch+1}/{num_epochs}] | Loss: {running_loss / len(train_loader):.4f}")

                scheduler.step()

            # Validation Metrics
            student_model.eval()
            all_labels = []
            all_preds = []

            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    student_logits = student_model(inputs)
                    _, preds = torch.max(student_logits, 1)
                    all_labels.extend(labels.cpu().numpy())
                    all_preds.extend(preds.cpu().numpy())

            accuracy = accuracy_score(all_labels, all_preds)
            print(f"Validation Accuracy: {accuracy:.4f}")



100%|██████████| 169M/169M [00:03<00:00, 50.9MB/s]


Extracting ./data/cifar-100-python.tar.gz to ./data


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 192MB/s]
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed i

Random Search | Model: ResNet50-ShuffleNetV1 | Temp: 3.85
Grid Search | Alpha: 0.1, Beta: 0.05, Temp: 3.85
Epoch [1/50] | Loss: 3.8620
Epoch [2/50] | Loss: 3.2718
Epoch [3/50] | Loss: 2.7991
Epoch [4/50] | Loss: 2.4658
Epoch [5/50] | Loss: 2.2059
Epoch [6/50] | Loss: 2.0261
Epoch [7/50] | Loss: 1.8798
Epoch [8/50] | Loss: 1.7577
Epoch [9/50] | Loss: 1.6496
Epoch [10/50] | Loss: 1.5644
Epoch [11/50] | Loss: 1.4843
Epoch [12/50] | Loss: 1.4157
Epoch [13/50] | Loss: 1.3427
Epoch [14/50] | Loss: 1.2871
Epoch [15/50] | Loss: 1.2281
Epoch [16/50] | Loss: 1.1782
Epoch [17/50] | Loss: 1.1272
Epoch [18/50] | Loss: 1.0776
Epoch [19/50] | Loss: 1.0343
Epoch [20/50] | Loss: 0.9938
Epoch [21/50] | Loss: 0.9485
Epoch [22/50] | Loss: 0.9069
Epoch [23/50] | Loss: 0.8730
Epoch [24/50] | Loss: 0.8365
Epoch [25/50] | Loss: 0.8046
Epoch [26/50] | Loss: 0.7646
Epoch [27/50] | Loss: 0.7358
Epoch [28/50] | Loss: 0.7081
Epoch [29/50] | Loss: 0.6753
Epoch [30/50] | Loss: 0.6539
Epoch [31/50] | Loss: 0.6227
Epo

KeyboardInterrupt: 

### Kindly Run this

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import accuracy_score
import itertools
import random

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
initial_alpha = 0.1
max_alpha = 0.9
initial_beta = 0.05
max_beta = 0.3
initial_temperature = 1
max_temperature = 5
learning_rate = 0.001
num_epochs = 50
batch_size = 64

# CIFAR-100 Data Transforms
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

transform_val = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load CIFAR-100 Dataset
dataset = datasets.CIFAR100(root='./data', train=True, transform=transform_train, download=True)

# Split Dataset (90% training, 10% validation)
train_size = 45000
val_size = 5000
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

# Knowledge Distillation Loss with Temperature Scaling
def custom_knowledge_distillation_loss(student_logits, teacher_logits, true_labels, alpha, beta, temperature):
    cross_entropy_loss = F.cross_entropy(student_logits, true_labels)
    teacher_probs = F.softmax(teacher_logits / temperature, dim=1)
    student_log_probs = F.log_softmax(student_logits / temperature, dim=1)
    distillation_loss = F.kl_div(student_log_probs, teacher_probs, reduction="batchmean") * (temperature ** 2)
    student_probs = F.softmax(student_logits, dim=1)
    entropy_regularization = -torch.sum(student_probs * torch.log(student_probs + 1e-8)) / student_probs.size(0)
    total_loss = (1 - alpha) * cross_entropy_loss + alpha * distillation_loss + beta * entropy_regularization
    return total_loss

# Random Search for Temperature
def random_search_temperature(num_samples):
    return [random.uniform(initial_temperature, max_temperature) for _ in range(num_samples)]

# Perform Randomized Search and Grid Search for Temperature
def random_search_temperature_and_grid_search(num_samples, top_k):
    # Step 1: Perform Randomized Search for Temperature
    temperature_candidates = random_search_temperature(num_samples)
    temp_results = []

    for temperature in temperature_candidates:
        print(f"Randomized Search | Temp: {temperature:.2f}")
        avg_accuracy = 0

        for alpha, beta in grid_search_params:
            optimizer = optim.AdamW(student_model.parameters(), lr=learning_rate)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

            for epoch in range(num_epochs // 5):  # Run fewer epochs for randomized search
                student_model.train()
                for inputs, labels in train_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    optimizer.zero_grad()

                    with torch.no_grad():
                        teacher_logits = teacher_model(inputs)
                    student_logits = student_model(inputs)

                    loss = custom_knowledge_distillation_loss(student_logits, teacher_logits, labels, alpha, beta, temperature)
                    loss.backward()
                    optimizer.step()

                scheduler.step()

            # Validation Metrics
            student_model.eval()
            all_labels = []
            all_preds = []
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    student_logits = student_model(inputs)
                    _, preds = torch.max(student_logits, 1)
                    all_labels.extend(labels.cpu().numpy())
                    all_preds.extend(preds.cpu().numpy())

            accuracy = accuracy_score(all_labels, all_preds)
            avg_accuracy += accuracy

        avg_accuracy /= len(grid_search_params)
        temp_results.append((temperature, avg_accuracy))

    # Step 2: Select Top-K Temperatures from Randomized Search
    temp_results.sort(key=lambda x: x[1], reverse=True)
    top_temperatures = [temp for temp, _ in temp_results[:top_k]]

    # Perform Grid Search for Hyperparameters with Top Temperatures
    best_params = None
    best_accuracy = 0

    for temperature in top_temperatures:
        print(f"Grid Search | Temp: {temperature:.2f}")
        for alpha, beta in grid_search_params:
            optimizer = optim.AdamW(student_model.parameters(), lr=learning_rate)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

            for epoch in range(num_epochs):
                student_model.train()
                for inputs, labels in train_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    optimizer.zero_grad()

                    with torch.no_grad():
                        teacher_logits = teacher_model(inputs)
                    student_logits = student_model(inputs)

                    loss = custom_knowledge_distillation_loss(student_logits, teacher_logits, labels, alpha, beta, temperature)
                    loss.backward()
                    optimizer.step()

                scheduler.step()

            # Validation Metrics
            student_model.eval()
            all_labels = []
            all_preds = []
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    student_logits = student_model(inputs)
                    _, preds = torch.max(student_logits, 1)
                    all_labels.extend(labels.cpu().numpy())
                    all_preds.extend(preds.cpu().numpy())

            accuracy = accuracy_score(all_labels, all_preds)
            if accuracy > best_accuracy:
                best_accuracy = accuracy
                best_params = {"temperature": temperature, "alpha": alpha, "beta": beta}

            print(f"Temp: {temperature:.2f} | Alpha: {alpha}, Beta: {beta} | Accuracy: {accuracy:.4f}")

    print(f"Best Params: {best_params} | Best Accuracy: {best_accuracy:.4f}")
    return best_params, best_accuracy


# Define teacher-student model pairs
model_pairs = {
    "ResNet50-ShuffleNetV1": (models.resnet50(pretrained=True), models.shufflenet_v2_x1_0(pretrained=False))
}

# Grid Search Parameters
alpha_values = [0.1, 0.3, 0.5]
beta_values = [0.05, 0.1, 0.2]
grid_search_params = list(itertools.product(alpha_values, beta_values))

# Train and Evaluate Models
for model_name, (teacher_model, student_model) in model_pairs.items():
    # Update the classifier layers
    teacher_model.fc = nn.Linear(teacher_model.fc.in_features, 100)
    student_model.fc = nn.Linear(student_model.fc.in_features, 100)

    teacher_model = teacher_model.to(device).eval()
    student_model = student_model.to(device)

    # Apply Randomized and Grid Search
    best_params, best_accuracy = random_search_temperature_and_grid_search(num_samples=10, top_k=3)
    print(f"Optimal Hyperparameters for {model_name}: {best_params} | Best Validation Accuracy: {best_accuracy:.4f}")


Files already downloaded and verified
Randomized Search | Temp: 3.18


KeyboardInterrupt: 